# <font color="#418FDE" size="6.5" uppercase>**Generalized Bases**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit Poisson, Gamma, and Tweedie regressors for appropriate target distributions. 
- Explain link functions and deviance through practical model evaluation. 
- Build polynomial and spline regression pipelines for nonlinear patterns. 


## **1. Generalized Regression Models**

### **1.1. Poisson Count Modeling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_01.jpg?v=1783994648" width="250">



>* Models nonnegative event counts over defined exposure
>* Predicts expected counts using relevant factors

>* Log links model multiplicative count changes
>* Exposure separates rates from opportunity

>* Check assumptions and watch for overdispersion
>* Use diagnostics before trusting Poisson results



In [ ]:
#@title Python Code - Poisson Count Modeling

# This example fits a Poisson count model.
# Counts are generated from exposure and advertising.
# The plot compares observed and predicted counts.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import train_test_split

# A fixed generator makes the simulated lesson repeatable.
rng = np.random.default_rng(42)

# Exposure means each row has different event opportunity.
n_samples = 300
exposure_hours = rng.uniform(1.0, 8.0, size=n_samples)
ad_spend = rng.uniform(0.0, 5.0, size=n_samples)

# The log link makes predictors multiply the expected count.
log_rate = -0.4 + 0.35 * ad_spend
expected_count = exposure_hours * np.exp(log_rate)
observed_count = rng.poisson(expected_count)

# The model learns from predictors, including exposure.
features = np.column_stack([ad_spend, np.log(exposure_hours)])
X_train, X_test, y_train, y_test = train_test_split(
    features, observed_count, test_size=0.25, random_state=42
)

# This validation keeps the count modeling assumption explicit.
if np.any(y_train < 0) or np.any(y_test < 0):
    raise ValueError("Poisson targets must be nonnegative counts.")

# PoissonRegressor uses a log link for positive predictions.
model = PoissonRegressor(alpha=0.0, max_iter=300)
model.fit(X_train, y_train)

# Deviance is a natural error measure for Poisson counts.
predicted_count = model.predict(X_test)
deviance = mean_poisson_deviance(y_test, predicted_count)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Mean observed test count: {np.mean(y_test):.2f}")
print(f"Mean predicted test count: {np.mean(predicted_count):.2f}")
print(f"Mean Poisson deviance: {deviance:.2f}")

# A perfect model would place points near the diagonal.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_test, predicted_count, alpha=0.7, label="test rows")

# The diagonal helps compare observed and predicted counts.
max_count = max(float(np.max(y_test)), float(np.max(predicted_count)))
ax.plot([0, max_count], [0, max_count], color="black", label="perfect match")
ax.set_title("Poisson regression for event counts")
ax.set_xlabel("Observed count")
ax.set_ylabel("Predicted expected count")
ax.legend()
plt.show()



### **1.2. Gamma Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_02.jpg?v=1783994652" width="250">



>* Models positive, continuous, right-skewed outcomes
>* Handles long tails better than linear regression

>* Larger predictions have greater uncertainty
>* Useful when variation scales with outcomes

>* Use Gamma for positive continuous targets
>* Check zeros, skew, and changing variance



In [ ]:
#@title Python Code - Gamma Regression

# This example fits a Gamma regression model.
# Gamma regression predicts positive skewed continuous outcomes.
# The plot compares true and predicted costs.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import GammaRegressor
from sklearn.metrics import mean_gamma_deviance
from sklearn.model_selection import train_test_split

# A fixed generator makes the synthetic lesson reproducible.
rng = np.random.default_rng(42)

# Create one feature representing service duration in hours.
n_samples = 300
hours = rng.uniform(0.5, 8.0, size=n_samples)

# The expected cost grows exponentially and stays positive.
expected_cost = np.exp(2.2 + 0.28 * hours)
shape = 8.0
cost = rng.gamma(shape=shape, scale=expected_cost / shape)

# Gamma regression requires strictly positive target values.
if np.min(cost) <= 0:
    raise ValueError("Gamma regression needs positive target values.")

# Split data so evaluation uses unseen examples.
X = hours.reshape(-1, 1)
y = cost
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit a Gamma model with the default log link.
model = GammaRegressor(alpha=0.0, max_iter=300)
model.fit(X_train, y_train)

# Predict positive costs and evaluate with Gamma deviance.
y_pred = model.predict(X_test)
deviance = mean_gamma_deviance(y_test, y_pred)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"All predictions are positive: {bool(np.all(y_pred > 0))}")
print(f"Mean Gamma deviance on test data: {deviance:.3f}")

# Sort test points so the fitted curve is easy to read.
order = np.argsort(X_test[:, 0])
sorted_hours = X_test[order, 0]
sorted_pred = y_pred[order]

# Show how Gamma regression follows positive skewed costs.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_test[:, 0], y_test, alpha=0.55, label="Observed test costs")
ax.plot(sorted_hours, sorted_pred, color="red", label="Gamma prediction")
ax.set_title("Gamma Regression for Positive Skewed Costs")
ax.set_xlabel("Service duration (hours)")
ax.set_ylabel("Cost (dollars)")
ax.legend()
plt.show()



### **1.3. Tweedie Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_01_03.jpg?v=1783994650" width="250">



>* Models zero-heavy, skewed positive outcomes
>* One framework for insurance, demand, energy

>* Tweedie bridges count and positive amount models
>* Models event occurrence and outcome size

>* Use Tweedie for zero-heavy continuous outcomes
>* Compare predictions against simpler model choices



In [ ]:
#@title Python Code - Tweedie Regression

# This example fits a Tweedie regression model.
# Tweedie targets can mix zeros and skewed positives.
# The output compares predictions with observed losses.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_tweedie_deviance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Generate a small zero-heavy continuous target.
rng = np.random.default_rng(42)
n_samples = 800

# Two features represent exposure and risk score.
exposure = rng.uniform(0.2, 2.0, n_samples)
risk_score = rng.normal(0.0, 1.0, n_samples)

# Higher risk increases event frequency and positive severity.
event_rate = exposure * np.exp(-1.0 + 0.8 * risk_score)
claim_count = rng.poisson(event_rate)

# Positive claims have continuous, right-skewed amounts.
severity_mean = np.exp(2.2 + 0.4 * risk_score)
claim_amount = rng.gamma(shape=2.0, scale=severity_mean / 2.0)

# Total loss is zero when no event occurs.
total_loss = claim_count * claim_amount
features = np.column_stack([exposure, risk_score])

# Validate the target pattern Tweedie regression is meant for.
zero_share = np.mean(total_loss == 0.0)
positive_share = np.mean(total_loss > 0.0)

if zero_share <= 0.0 or positive_share <= 0.0:
    raise ValueError("The generated target must contain zeros and positives.")

# Split data before fitting the scaler and model.
X_train, X_test, y_train, y_test = train_test_split(
    features, total_loss, test_size=0.25, random_state=42
)

# Power between one and two models zero-heavy positive outcomes.
model = make_pipeline(
    StandardScaler(),
    TweedieRegressor(power=1.5, alpha=0.0, max_iter=1000)
)

model.fit(X_train, y_train)
predicted_loss = model.predict(X_test)

# Tweedie deviance rewards predictions matching this distributional shape.
deviance = mean_tweedie_deviance(y_test, predicted_loss, power=1.5)
mean_observed = np.mean(y_test)
mean_predicted = np.mean(predicted_loss)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Zero outcomes in full data: {zero_share:.1%}")
print(f"Mean observed test loss: {mean_observed:.2f}")
print(f"Mean predicted test loss: {mean_predicted:.2f}")
print(f"Test Tweedie deviance: {deviance:.2f}")

# Plot expected loss against risk score for the test set.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_test[:, 1], y_test, s=18, alpha=0.35, label="Observed loss")
ax.scatter(X_test[:, 1], predicted_loss, s=18, alpha=0.65, label="Predicted mean")
ax.set_title("Tweedie Regression for Zero-Heavy Losses")
ax.set_xlabel("Risk score")
ax.set_ylabel("Loss amount")
ax.legend()
plt.show()



## **2. GLM Evaluation**

### **2.1. Link Function Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_01.jpg?v=1783994661" width="250">



>* Link functions connect linear models to outcomes
>* Features add effects on transformed scales

>* Links shape fitting and coefficient meaning
>* Back-transform predictions for real-world interpretation

>* Judge models on their linked scale
>* Check transformed predictions and error patterns



### **2.2. Deviance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_02.jpg?v=1783994662" width="250">



>* Deviance measures GLM lack of fit
>* Lower deviance means better distribution-aware predictions

>* Deviance depends on the target distribution
>* Compare models using real outcome patterns

>* Compare deviance across models and validation data
>* Use deviance residuals to diagnose fit issues



In [ ]:
#@title Python Code - Deviance Metrics

# This example compares Poisson deviance scores.
# Deviance measures distribution-aware prediction error.
# Lower test deviance indicates better count predictions.

import numpy as np
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import train_test_split

# Create a small count dataset with a log link pattern.
rng = np.random.default_rng(42)
feature = rng.uniform(0.0, 3.0, size=300)

# The Poisson mean must stay positive.
true_mean = np.exp(0.4 + 0.7 * feature)
counts = rng.poisson(true_mean)

# Scikit-learn expects a two-dimensional feature matrix.
X = feature.reshape(-1, 1)
y = counts

# Validate the target before fitting a Poisson model.
if np.any(y < 0):
    raise ValueError("Poisson targets must be non-negative counts.")

# Split data so deviance measures generalization.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Fit one GLM and one simple baseline.
poisson_model = PoissonRegressor(alpha=0.0, max_iter=300)
poisson_model.fit(X_train, y_train)

baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

# Predict positive means for the Poisson deviance formula.
poisson_predictions = poisson_model.predict(X_test)
baseline_predictions = baseline_model.predict(X_test)

# Compare models using distribution-aware deviance.
poisson_deviance = mean_poisson_deviance(y_test, poisson_predictions)
baseline_deviance = mean_poisson_deviance(y_test, baseline_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Baseline mean Poisson deviance: {baseline_deviance:.3f}")
print(f"Poisson GLM mean Poisson deviance: {poisson_deviance:.3f}")
print("Lower deviance means predictions fit the count distribution better.")



### **2.3. Assessing Model Fit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_02_03.jpg?v=1783994664" width="250">



>* Match GLM family to target distribution
>* Check prediction errors across ranges and groups

>* Deviance compares distribution-aware model fit
>* Balance deviance with validation and interpretability

>* Check residuals across important data segments
>* Validate calibration and iterate model choices



In [ ]:
#@title Python Code - Assessing Model Fit

# This example evaluates a Poisson generalized linear model.
# Deviance compares distribution-aware fit against a baseline.
# Calibration checks whether grouped predictions match observations.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import train_test_split

# Synthetic counts keep the lesson small and reproducible.
rng = np.random.default_rng(42)
n_samples = 600
company_size = rng.uniform(1.0, 10.0, size=n_samples)

# The log link makes the expected count stay positive.
log_expected_tickets = 0.4 + 0.18 * company_size
expected_tickets = np.exp(log_expected_tickets)
tickets = rng.poisson(expected_tickets)

# The feature matrix must be two-dimensional for scikit-learn.
X = company_size.reshape(-1, 1)
y = tickets

# A quick validation catches shape mistakes before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target values must match.")

# Split data so deviance measures performance on unseen cases.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# PoissonRegressor uses a log link for nonnegative count outcomes.
model = PoissonRegressor(alpha=0.0, max_iter=300)
model.fit(X_train, y_train)

# Compare the fitted model with a simple mean-only baseline.
predicted = model.predict(X_test)
baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

# Mean Poisson deviance is lower when count predictions fit better.
model_deviance = mean_poisson_deviance(y_test, predicted)
baseline_deviance = mean_poisson_deviance(y_test, baseline)

# Calibration groups cases with similar predicted counts.
order = np.argsort(predicted)
groups = np.array_split(order, 5)
calibration_x = []
calibration_y = []

for group in groups:
    calibration_x.append(predicted[group].mean())
    calibration_y.append(y_test[group].mean())

print("scikit-learn version:", sklearn.__version__)
print("Baseline mean Poisson deviance:", round(baseline_deviance, 3))
print("Model mean Poisson deviance:", round(model_deviance, 3))
print("Lower deviance means better distribution-aware fit.")

# The diagonal line represents perfect average calibration.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(calibration_x, calibration_y, marker="o", label="Grouped test data")
ax.plot([min(calibration_x), max(calibration_x)], [min(calibration_x), max(calibration_x)], label="Perfect calibration")
ax.set_title("Poisson GLM Calibration Check")
ax.set_xlabel("Mean predicted ticket count")
ax.set_ylabel("Mean observed ticket count")
ax.legend()
plt.show()



## **3. Basis Design**

### **3.1. Polynomial Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_01.jpg?v=1783994654" width="250">



>* Add powers of inputs to model curves
>* Capture nonlinear patterns while staying interpretable

>* Higher degrees add flexibility and overfitting risk
>* Use validation and regularization for smooth generalization

>* Transform and scale features before fitting
>* Capture curvature and interactions transparently



In [ ]:
#@title Python Code - Polynomial Features

# This example fits curved data with polynomial features.
# A pipeline prevents preprocessing leakage during training.
# The plot compares linear and polynomial predictions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

# Create a small nonlinear regression dataset.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 80).reshape(-1, 1)
noise = rng.normal(0, 2.0, size=x.shape[0])
y = 3 + 1.5 * x.ravel() - 0.35 * x.ravel() ** 2 + noise

# Check that features and targets align.
if x.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Split data before fitting any preprocessing step.
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Fit a straight-line baseline model.
linear_model = LinearRegression()
linear_model.fit(x_train, y_train)
linear_predictions = linear_model.predict(x_test)

# Fit polynomial features inside a safe pipeline.
polynomial_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LinearRegression(),
)
polynomial_model.fit(x_train, y_train)
polynomial_predictions = polynomial_model.predict(x_test)

# Print concise evaluation results.
linear_rmse = mean_squared_error(y_test, linear_predictions) ** 0.5
polynomial_rmse = mean_squared_error(y_test, polynomial_predictions) ** 0.5
print("scikit-learn version:", sklearn.__version__)
print("Linear test RMSE:", round(linear_rmse, 2))
print("Polynomial degree 2 test RMSE:", round(polynomial_rmse, 2))

# Predict smooth curves for visual comparison.
x_grid = np.linspace(0, 10, 200).reshape(-1, 1)
linear_curve = linear_model.predict(x_grid)
polynomial_curve = polynomial_model.predict(x_grid)

# Plot the data and both fitted relationships.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x_train, y_train, label="training data", alpha=0.7)
ax.plot(x_grid, linear_curve, label="linear model", linewidth=2)
ax.plot(x_grid, polynomial_curve, label="degree 2 polynomial", linewidth=2)
ax.set_title("Polynomial Features Add Curvature to Linear Regression")
ax.set_xlabel("Input value")
ax.set_ylabel("Target value")
ax.legend()
plt.show()



### **3.2. Spline Basis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_02.jpg?v=1783994659" width="250">



>* Splines join smooth local curve pieces
>* Knots help capture changing nonlinear patterns

>* Choose knots to balance flexibility and stability
>* Capture nonlinear patterns while preserving generalization

>* Spline features enable nonlinear linear-model predictions
>* Fit transformations on training data only



In [ ]:
#@title Python Code - Spline Basis

# This example builds a spline regression pipeline.
# Spline features let linear models bend smoothly.
# The plot compares data with spline predictions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

# Create a small nonlinear regression dataset.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 120)
noise = rng.normal(0, 0.35, size=x.shape)
y = 2 + np.sin(x) + 0.25 * x + noise

# Scikit-learn expects a two-dimensional feature matrix.
X = x.reshape(-1, 1)
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Build one pipeline with spline features and linear regression.
spline_model = make_pipeline(
    SplineTransformer(n_knots=6, degree=3, include_bias=False),
    LinearRegression(),
)

# Fit the pipeline on the original one-column input.
spline_model.fit(X, y)
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)
y_grid = spline_model.predict(x_grid)

# Show how one input becomes several spline basis columns.
spline_step = spline_model.named_steps["splinetransformer"]
basis_columns = spline_step.transform(X[:1]).shape[1]
print("scikit-learn version:", sklearn.__version__)
print("Original feature columns: 1")
print("Spline basis columns:", basis_columns)

# Plot the noisy data and the smooth spline fit.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, s=22, alpha=0.65, label="observed data")
ax.plot(x_grid.ravel(), y_grid, color="crimson", linewidth=2.5, label="spline fit")
ax.set_title("Spline basis regression captures a smooth nonlinear pattern")
ax.set_xlabel("Input feature")
ax.set_ylabel("Target value")
ax.legend()
plt.show()



### **3.3. Interaction Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_B/image_03_03.jpg?v=1783994656" width="250">



>* Interactions model context-dependent predictor effects
>* They modify nonlinear curves across conditions

>* Create interactions for purposeful joint patterns
>* Match interaction complexity to available data

>* Too many interactions can overfit data
>* Validate, regularize, and scale for stability



In [ ]:
#@title Python Code - Interaction Features

# This example shows why interaction features matter.
# Multiplication lets one predictor change another effect.
# The plot compares additive and interaction pipelines.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

# A fixed generator makes the synthetic data repeatable.
rng = np.random.default_rng(42)

# Study hours and preparation jointly affect the score.
study_hours = rng.uniform(0, 10, 180)
preparation = rng.uniform(0, 1, 180)
noise = rng.normal(0, 4, 180)

# The true pattern includes a study by preparation interaction.
score = 45 + 2 * study_hours + 8 * preparation
score = score + 3 * study_hours * preparation + noise

# The feature matrix has two original predictors.
X = np.column_stack((study_hours, preparation))
y = score

# Validate the basic shape before modeling.
if X.shape != (180, 2):
    raise ValueError("Expected 180 rows and 2 features.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# This model can only add separate feature effects.
additive_model = LinearRegression()
additive_model.fit(X_train, y_train)
additive_predictions = additive_model.predict(X_test)

# This pipeline adds x1*x2 without adding squared terms.
interaction_model = make_pipeline(
    PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
    LinearRegression(),
)
interaction_model.fit(X_train, y_train)
interaction_predictions = interaction_model.predict(X_test)

# Compare test errors for the two basis designs.
additive_rmse = np.sqrt(mean_squared_error(y_test, additive_predictions))
interaction_rmse = np.sqrt(mean_squared_error(y_test, interaction_predictions))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Additive test RMSE: {additive_rmse:.2f} score points")
print(f"Interaction test RMSE: {interaction_rmse:.2f} score points")

# Plot predictions as preparation changes the study effect.
grid_hours = np.linspace(0, 10, 100)
low_prep = np.column_stack((grid_hours, np.full(100, 0.2)))
high_prep = np.column_stack((grid_hours, np.full(100, 0.8)))

low_curve = interaction_model.predict(low_prep)
high_curve = interaction_model.predict(high_prep)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(study_hours, score, s=18, alpha=0.35, label="observed students")
ax.plot(grid_hours, low_curve, label="prep = 0.2")
ax.plot(grid_hours, high_curve, label="prep = 0.8")
ax.set_title("Interaction feature changes the study-hours slope")
ax.set_xlabel("Study hours")
ax.set_ylabel("Exam score")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Generalized Bases**</font>


In this lecture, you learned to:
- Fit Poisson, Gamma, and Tweedie regressors for appropriate target distributions. 
- Explain link functions and deviance through practical model evaluation. 
- Build polynomial and spline regression pipelines for nonlinear patterns. 

In the next Module (Module 11), we will go over 'Linear Classification'